# Geographical Augmentation with Country Matcher

This notebook reads the parquet data and enriches it with geographical information using the country_matcher module.

In [1]:
import sys
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Add the 6_Method3 directory to the path to import country_matcher
sys.path.insert(0, '6_Method3')

from country_matcher import extract_place_struct, add_location_columns

print("✓ Modules imported successfully")

✓ Modules imported successfully


## Load Parquet Data

In [2]:
# Read the parquet data
data_path = 'data/weekly_aggregated_by_week_domain_keyword (1).parquet'
df = pd.read_parquet(data_path)

print(f"✓ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

✓ Data loaded: 199,272 rows × 26 columns

Columns: ['week', 'domain', 'keyword', 'impressions_sum', 'adcost_sum', 'adclicks_sum', 'cpc_week', 'avg_sim_top25_this_week', 'avg_sim_top25_last_week', 'n_sim_this_week', 'n_sim_last_week', 'dom_share_avis', 'dom_share_avisautonoleggio', 'dom_share_aviscarsales', 'dom_share_budget', 'dom_share_dollar', 'dom_share_economybookings', 'dom_share_hertz', 'dom_share_letsdrive', 'dom_share_sixt', 'dom_share_thrifty', 'n_dev_desktop', 'n_dev_mobile', 'n_dev_tablet', 'n_st_branded_search', 'n_st_generic_search']

First few rows:


,week,domain,keyword,impressions_sum,adcost_sum,adclicks_sum,cpc_week,avg_sim_top25_this_week,avg_sim_top25_last_week,n_sim_this_week,...,dom_share_economybookings,dom_share_hertz,dom_share_letsdrive,dom_share_sixt,dom_share_thrifty,n_dev_desktop,n_dev_mobile,n_dev_tablet,n_st_branded_search,n_st_generic_search
0,01-2021,avis,3 month hire,5.0,3.399534,6.0,0.566589,0.308505,0.310740,33357,...,0.357203,0.024431,0.0,0.406066,0.029486,2,3,0,0,5
1,01-2021,avis,airport aberdeen hire,4.0,8.208105,4.0,2.052026,0.390090,0.381570,4320,...,0.357203,0.024431,0.0,0.406066,0.029486,2,2,0,4,0
2,01-2021,avis,airport avis hire aberdeen,4.0,7.243941,4.0,1.810985,0.451481,0.449888,11694,...,0.357203,0.024431,0.0,0.406066,0.029486,0,4,0,4,0
3,01-2021,avis,airport avis hire birmingham,2.0,7.904708,2.0,3.952354,0.465347,0.466825,11946,...,0.357203,0.024431,0.0,0.406066,0.029486,0,2,0,2,0
4,01-2021,avis,airport avis hire glasgow,4.0,21.599380,4.0,5.399845,0.472526,0.472557,11773,...,0.357203,0.024431,0.0,0.406066,0.029486,0,4,0,4,0


In [3]:
# Check data info
print("Data Info:")
print(df.info())
print(f"\n{'='*50}\n")
print("Data description:")
print(df.describe())

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199272 entries, 0 to 199271
Data columns (total 26 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   week                        199272 non-null  object 
 1   domain                      199272 non-null  object 
 2   keyword                     199272 non-null  object 
 3   impressions_sum             199272 non-null  float64
 4   adcost_sum                  199272 non-null  float64
 5   adclicks_sum                199272 non-null  float64
 6   cpc_week                    199272 non-null  float64
 7   avg_sim_top25_this_week     199272 non-null  float64
 8   avg_sim_top25_last_week     198257 non-null  float64
 9   n_sim_this_week             199272 non-null  int64  
 10  n_sim_last_week             199272 non-null  int64  
 11  dom_share_avis              199272 non-null  float64
 12  dom_share_avisautonoleggio  199272 non-null  float64
 13  dom

## Test Location Extraction on Sample Keywords

First, let's test the location extraction on a few sample keywords from the data.

In [4]:
# Check if there's a keyword column and get unique keywords
if 'keyword' in df.columns:
    unique_keywords = df['keyword'].unique()
    sample_keywords = unique_keywords[:10]
    print(f"Total unique keywords: {len(unique_keywords):,}")
    print(f"\nSample keywords to test:")
    for i, kw in enumerate(sample_keywords, 1):
        print(f"{i}. {kw}")
else:
    print("Available columns:", list(df.columns))
    print("\nNote: Looking for a 'keyword' column. You may need to rename a column.")

Total unique keywords: 1,519

Sample keywords to test:
1. 3 month hire
2. airport aberdeen hire
3. airport avis hire aberdeen
4. airport avis hire birmingham
5. airport avis hire glasgow
6. airport avis hire manchester
7. airport avis hire newcastle
8. airport edinburgh hire
9. airport hire
10. airport hire aberdeen


In [5]:
# Test location extraction on sample keywords
if 'keyword' in df.columns:
    print("Testing location extraction on sample keywords:\n")
    print(f"{'Keyword':<50} {'City':<20} {'Country':<25} {'Continent':<20}")
    print("="*115)
    
    for kw in sample_keywords:
        result = extract_place_struct(str(kw))
        city = result['detected_city'] or '-'
        country = result['detected_country']
        continent = result['detected_continent']
        print(f"{str(kw)[:48]:<50} {city:<20} {country:<25} {continent:<20}")

Testing location extraction on sample keywords:

Keyword                                            City                 Country                   Continent           
3 month hire                                       -                    Global                    Global              
airport aberdeen hire                              Aberdeen             United Kingdom            Europe              
airport avis hire aberdeen                         Aberdeen             United Kingdom            Europe              
airport avis hire birmingham                       Birmingham           United Kingdom            Europe              
airport avis hire glasgow                          Glasgow              United Kingdom            Europe              
airport avis hire manchester                       Manchester           United Kingdom            Europe              
airport avis hire newcastle                        Newcastle            Australia                 Oceania             

## Enrich Full Dataset with Geographical Information

Now let's enrich the entire dataset by adding geographical columns.

In [6]:
# Progress tracking function
def progress_tracker(current, total):
    percentage = (current / total) * 100
    if current % 100 == 0 or current == total:  # Print every 100 keywords
        print(f"Progress: {current:,}/{total:,} keywords ({percentage:.1f}%)")

# Enrich the dataset with geographical information
if 'keyword' in df.columns:
    print(f"Enriching {len(df['keyword'].unique()):,} unique keywords with geographical data...\n")
    
    df_enriched = add_location_columns(
        df,
        keywords=df['keyword'].unique().tolist(),
        progress_callback=progress_tracker,
        batch_size=100
    )
    
    print("\n✓ Enrichment complete!")
    print(f"\nNew columns added: detected_city, detected_country, detected_continent")
    print(f"Enriched data shape: {df_enriched.shape}")
else:
    print("Cannot proceed: 'keyword' column not found in dataframe")

Enriching 1,519 unique keywords with geographical data...

Progress: 100/1,519 keywords (6.6%)
Progress: 200/1,519 keywords (13.2%)
Progress: 300/1,519 keywords (19.7%)
Progress: 400/1,519 keywords (26.3%)
Progress: 500/1,519 keywords (32.9%)
Progress: 600/1,519 keywords (39.5%)
Progress: 700/1,519 keywords (46.1%)
Progress: 800/1,519 keywords (52.7%)
Progress: 900/1,519 keywords (59.2%)
Progress: 1,000/1,519 keywords (65.8%)
Progress: 1,100/1,519 keywords (72.4%)
Progress: 1,200/1,519 keywords (79.0%)
Progress: 1,300/1,519 keywords (85.6%)
Progress: 1,400/1,519 keywords (92.2%)
Progress: 1,500/1,519 keywords (98.7%)
Progress: 1,519/1,519 keywords (100.0%)

✓ Enrichment complete!

New columns added: detected_city, detected_country, detected_continent
Enriched data shape: (199272, 29)


## Analyze Enriched Data

In [7]:
# Display sample of enriched data
if 'keyword' in df.columns:
    print("Sample of enriched data:")
    display_cols = ['keyword', 'detected_city', 'detected_country', 'detected_continent']
    display_cols = [col for col in display_cols if col in df_enriched.columns]
    df_enriched[display_cols].head(20)

Sample of enriched data:


In [8]:
# Geographical distribution statistics
if 'keyword' in df.columns and 'detected_continent' in df_enriched.columns:
    print("GEOGRAPHICAL DISTRIBUTION\n")
    
    print("By Continent:")
    print(df_enriched['detected_continent'].value_counts())
    
    print("\n" + "="*60 + "\n")
    
    print("Top 20 Countries:")
    print(df_enriched['detected_country'].value_counts().head(20))
    
    print("\n" + "="*60 + "\n")
    
    print("Top 20 Cities (with detected cities):")
    print(df_enriched[df_enriched['detected_city'].notna()]['detected_city'].value_counts().head(20))

GEOGRAPHICAL DISTRIBUTION

By Continent:
detected_continent
North America    76556
Global           68753
Europe           35373
Africa            7978
Oceania           7110
Asia              2532
South America      970
Name: count, dtype: int64


Top 20 Countries:
detected_country
United States           69286
Global                  68753
United Kingdom          13924
Mali                     6455
Australia                5856
Italy                    4854
Canada                   4527
France                   3097
Germany                  2291
Spain                    2148
Portugal                 1605
Austria                  1433
South Africa             1273
New Zealand              1254
United Arab Emirates     1107
Croatia                   831
Mexico                    818
Cuba                      723
Philippines               593
Netherlands               478
Name: count, dtype: int64


Top 20 Cities (with detected cities):
detected_city
San                6455
Orlando     

In [9]:
# Coverage statistics
if 'keyword' in df.columns and 'detected_city' in df_enriched.columns:
    total_keywords = len(df_enriched['keyword'].unique())
    with_city = df_enriched[df_enriched['detected_city'].notna()]['keyword'].nunique()
    with_country = df_enriched[df_enriched['detected_country'] != 'Global']['keyword'].nunique()
    
    print("COVERAGE STATISTICS\n")
    print(f"Total unique keywords: {total_keywords:,}")
    print(f"Keywords with detected city: {with_city:,} ({with_city/total_keywords*100:.1f}%)")
    print(f"Keywords with detected country: {with_country:,} ({with_country/total_keywords*100:.1f}%)")
    print(f"Keywords without geographical info: {total_keywords - with_country:,} ({(total_keywords-with_country)/total_keywords*100:.1f}%)")

COVERAGE STATISTICS

Total unique keywords: 1,519
Keywords with detected city: 923 (60.8%)
Keywords with detected country: 1,005 (66.2%)
Keywords without geographical info: 514 (33.8%)


## Example Filters and Queries

In [10]:
# Example filters and queries
if 'keyword' in df.columns and 'detected_country' in df_enriched.columns:
    print("EXAMPLE 1: Keywords from United States")
    us_keywords = df_enriched[df_enriched['detected_country'] == 'United States']['keyword'].unique()
    print(f"Found {len(us_keywords):,} unique US keywords")
    print("Sample:", us_keywords[:10].tolist())
    
    print("\n" + "="*60 + "\n")
    
    print("EXAMPLE 2: Keywords from Europe")
    europe_keywords = df_enriched[df_enriched['detected_continent'] == 'Europe']['keyword'].unique()
    print(f"Found {len(europe_keywords):,} unique European keywords")
    print("Sample:", europe_keywords[:10].tolist())
    
    print("\n" + "="*60 + "\n")
    
    print("EXAMPLE 3: Keywords with specific cities")
    city_data = df_enriched[df_enriched['detected_city'].notna()][['keyword', 'detected_city', 'detected_country']].drop_duplicates()
    print(f"Found {len(city_data):,} keywords with city information")
    print("\nSample:")
    print(city_data.head(15))

EXAMPLE 1: Keywords from United States
Found 517 unique US keywords
Sample: ['airport hire geneva', 'avis rent near me', 'avis rentals near me', 'car rental avis chicago', 'car rental avis denver', 'car rental avis las vegas', 'car rental avis memphis tn', 'car rental avis miami', 'car rental avis near me', 'car rental avis orlando']


EXAMPLE 2: Keywords from Europe
Found 276 unique European keywords
Sample: ['airport aberdeen hire', 'airport avis hire aberdeen', 'airport avis hire birmingham', 'airport avis hire glasgow', 'airport avis hire manchester', 'airport edinburgh hire', 'airport hire aberdeen', 'airport hire birmingham', 'airport hire edinburgh', 'airport hire glasgow']


EXAMPLE 3: Keywords with specific cities
Found 923 keywords with city information

Sample:
                         keyword detected_city detected_country
1          airport aberdeen hire      Aberdeen   United Kingdom
2     airport avis hire aberdeen      Aberdeen   United Kingdom
3   airport avis hire bir

## Save Enriched Data

In [ ]:
# Save the enriched data
if 'keyword' in df.columns:
    output_path = 'data/weekly_aggregated_geo_enriched.parquet'
    df_enriched.to_parquet(output_path, index=False)
    print(f"✓ Enriched data saved to: {output_path}")
    print(f"  Shape: {df_enriched.shape}")
    print(f"  Columns: {list(df_enriched.columns)}")
    


✓ Enriched data saved to: data/weekly_aggregated_geo_enriched.parquet
  Shape: (199272, 29)
  Columns: ['week', 'domain', 'keyword', 'impressions_sum', 'adcost_sum', 'adclicks_sum', 'cpc_week', 'avg_sim_top25_this_week', 'avg_sim_top25_last_week', 'n_sim_this_week', 'n_sim_last_week', 'dom_share_avis', 'dom_share_avisautonoleggio', 'dom_share_aviscarsales', 'dom_share_budget', 'dom_share_dollar', 'dom_share_economybookings', 'dom_share_hertz', 'dom_share_letsdrive', 'dom_share_sixt', 'dom_share_thrifty', 'n_dev_desktop', 'n_dev_mobile', 'n_dev_tablet', 'n_st_branded_search', 'n_st_generic_search', 'detected_city', 'detected_country', 'detected_continent']

✓ Also saved as CSV: data/weekly_aggregated_geo_enriched.csv
